# Chuyên đề 02 · Kiểm chứng nghiên cứu mới về huấn luyện: Adam và lớp hiếm, rìa ổn định, grokking

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd02-nghien-cuu-huan-luyen.ipynb)

Notebook đi kèm [Chuyên đề 02 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/nghien-cuu-huan-luyen/). Ba kết quả nghiên cứu gần đây về cách mạng nơ-ron
được huấn luyện, mỗi kết quả kiểm lại trên một bài toán nhỏ đủ để chạy trên CPU. Ta không tin lời bài báo: ta đặt
lại thí nghiệm, chạy nhiều seed, và ghi rõ kết quả **khớp**, **không khớp** hay **tùy điều kiện**.

**Bạn sẽ làm:**
1. Thấy vì sao Adam thắng gradient descent khi các lớp có tần suất rất chênh lệch (Kunstner và cộng sự, 2024).
2. Đo độ cong lớn nhất của hàm mất mát trong lúc huấn luyện, và thấy nó dừng đúng ở $2/\eta$: "rìa ổn định"
   (Cohen và cộng sự, 2021).
3. Tái hiện grokking trên phép cộng modulo 113 và kiểm lời giải thích "sụp đổ softmax" (Prieto và cộng sự, 2025).

**Cần biết trước:** [Bài 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gradient-descent/)
(bước ổn định khi $\eta < 2/\lambda$), [Bài 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/)
và [Bài 08 · Quá khớp & chính quy hóa](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/chong-qua-khop/).
**Thời gian chạy:** khoảng 1 phút trên CPU, kể cả tải MNIST lần đầu.

Như Chuyên đề 01, notebook này dùng **PyTorch**. Mỗi mục chạy trực tiếp một phiên bản ngắn, rồi nạp kết quả đầy đủ
(nhiều seed, nhiều tốc độ học, chạy lâu hơn) mà các script `reel/research/*.py` của khóa học đã tính sẵn. Phần chạy
trực tiếp dùng đúng mã và đúng seed của các script đó, nên notebook kiểm được rằng nó ra cùng con số.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

PyTorch, numpy và matplotlib (Colab đã cài sẵn). Ta dùng 2 luồng CPU như máy Colab miễn phí và như các script: số
luồng khác thì phép cộng dấu phẩy động cộng theo thứ tự khác, và các chữ số cuối lệch đi. `vn()` in số kiểu Việt Nam
(dấu phẩy thập phân); `num()` in số rất nhỏ hay rất lớn dạng $3{,}3 \cdot 10^{-9}$.

In [ ]:
import gzip
import hashlib
import json
import math
import os
import urllib.request
from itertools import pairwise
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=1):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def num(x, sig=3):
    """sig chữ số có nghĩa: 0,243 · 0,0884 · 55,7 · 6,60·10⁻³"""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    if -2 <= e < 4:
        return vn(x, max(0, sig - 1 - e))
    return f"{vn(x / 10**e, sig - 1)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Nhãn trục kiểu Việt Nam; gọi ngay trước plt.show(). Trục log ngắn lấy vạch 1–2–5, trục log dài lấy lũy thừa của 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))

## 2 · Dữ liệu

Mục 3 dùng 1.000 ảnh MNIST đầu tiên (tải hai tệp huấn luyện, có kiểm MD5). Ba tệp kết quả của các script được nạp
bằng `fetch_json`, cũng có kiểm MD5.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
}
DATA = Path("mnist")
DATA.mkdir(exist_ok=True)
for name, md5 in FILES.items():
    p = DATA / name
    if not p.exists():
        urllib.request.urlretrieve(URL + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


TAIL = fetch_json("research-adam-tail.json", "bf394b41a981e52d6208adbc82860f12")
EOS = fetch_json("research-eos.json", "190b3db8f62a01b99e46c6b9bf237c43")
GROK = fetch_json("research-grok.json", "6a50b726ed7d60635a3262c1a5da4621")

## 3 · Vì sao Adam thắng gradient descent trên các lớp hiếm

**Nhận định** (Kunstner và cộng sự, 2024): khi tần suất các lớp rất chênh lệch — như từ vựng của ngôn ngữ, nơi vài
từ chiếm phần lớn văn bản và hàng chục nghìn từ hiếm — gradient descent học các lớp thường gặp nhanh và các lớp hiếm
rất chậm, còn Adam (và sign descent, tức đi theo dấu của gradient) tiến đều ở mọi lớp. Theo họ, đây là một phần lớn
lý do Adam vượt SGD khi huấn luyện mô hình ngôn ngữ. Hai bài sau (Kunstner & Bach, 2025; Yadav và cộng sự, 2025)
chứng minh điều này trên các mô hình tuyến tính nhỏ.

Ta dùng đúng bài toán tổng hợp của bài báo (phụ lục A.2), cỡ nhỏ: 7 **nhóm tần suất**. Nhóm 1 có 1 lớp với 128 mẫu,
nhóm 2 có 2 lớp với 64 mẫu mỗi lớp, …, nhóm 7 có 64 lớp với 2 mẫu mỗi lớp: tần suất giảm một nửa còn số lớp gấp
đôi, nên mỗi nhóm có đúng 128 mẫu. Tổng cộng 896 mẫu, 127 lớp. Đầu vào là 1.024 số ngẫu nhiên đều trong $[0, 1]$,
**không liên quan gì đến nhãn**: không có gì để "hiểu", nhưng có nhiều chiều hơn số mẫu nên mô hình tuyến tính
$z = xW + b$ vẫn khớp được hết. Huấn luyện toàn lô, bắt đầu từ $W = 0$.

In [ ]:
M_GROUPS = 7


def heavy_tailed(seed=0):
    """Bài toán của Kunstner và cộng sự (phụ lục A.2), giống hệt reel/research/adam_tail.py."""
    g = torch.Generator().manual_seed(1000 + seed)
    ys, gs, k = [], [], 0
    for i in range(1, M_GROUPS + 1):
        for _ in range(2 ** (i - 1)):
            ys += [k] * 2 ** (M_GROUPS - i + 1)
            gs += [i - 1] * 2 ** (M_GROUPS - i + 1)
            k += 1
    X = torch.rand(len(ys), (M_GROUPS + 1) * 2**M_GROUPS, generator=g)
    return X, torch.tensor(ys), torch.tensor(gs), k


def train_linear(opt_name, eta, steps, seed=0):
    """Mất mát trung bình của từng nhóm tần suất sau `steps` bước."""
    X, y, group, c = heavy_tailed(seed)
    W = torch.zeros(X.shape[1], c, requires_grad=True)
    b = torch.zeros(c, requires_grad=True)
    opt = {"gd": torch.optim.SGD([W, b], lr=eta), "adam": torch.optim.Adam([W, b], lr=eta), "sign": None}[opt_name]
    for _ in range(steps):
        W.grad = b.grad = None
        nn.functional.cross_entropy(X @ W + b, y).backward()
        if opt is None:  # sign descent: mỗi tham số đi một bước η theo dấu gradient của nó
            with torch.no_grad():
                W -= eta * W.grad.sign()
                b -= eta * b.grad.sign()
        else:
            opt.step()
    with torch.no_grad():
        per = nn.functional.cross_entropy(X @ W + b, y, reduction="none")
    return [per[group == k].mean().item() for k in range(M_GROUPS)]


X, y, group, n_cls = heavy_tailed()
print(f"{vn(len(y), 0)} mẫu, {n_cls} lớp, {vn(X.shape[1], 0)} chiều; mẫu mỗi nhóm: {torch.bincount(group).tolist()}")
print(f"mất mát lúc đầu (W = 0) = ln {n_cls} = {vn(math.log(n_cls), 3)}")

Mỗi bộ tối ưu dùng tốc độ học mà script đã chọn cho nó theo đúng quy tắc của bài báo: trên một lưới các lũy thừa
$10^{k/2}$, lấy $\eta$ cho mất mát cuối **tệ nhất trong 3 seed** nhỏ nhất. Dữ liệu tách được, nên một $\eta$ khổng lồ có
thể làm mất mát vọt lên hàng nghìn rồi vẫn về gần 0; bài báo khi đó lấy "bước nhỏ hơn kế tiếp", còn script ghi rõ
quy tắc: chỉ tính các $\eta$ mà mất mát không bao giờ vượt mức ban đầu $\ln 127$.

### 🤔 Dự đoán trước
Sau 300 bước, với gradient descent, nhóm nào có mất mát cao nhất: nhóm 1 (lớp phổ biến nhất) hay nhóm 7 (các lớp chỉ
có 2 mẫu)? Với Adam thì sao?

In [ ]:
best = TAIL["heavy"]["best"]
live = {o: train_linear(o, best[o], 300) for o in ("gd", "sign", "adam")}
names = {"gd": "gradient descent", "sign": "sign descent", "adam": "Adam"}
print("mất mát theo nhóm tần suất sau 300 bước (nhóm 1 = phổ biến nhất)")
for o, v in live.items():
    print(f"{names[o]:16} η = {num(best[o], 2):>6}: " + "  ".join(vn(x, 3) for x in v))
i300 = TAIL["check"].index(300)
for o, v in live.items():
    stored = TAIL["heavy"]["runs"][o][0][i300]
    assert all(abs(a - b) <= 2e-3 * max(1.0, b) for a, b in zip(v, stored)), f"{o}: khác với kết quả đã lưu"

Gradient descent: mất mát **tăng dần theo độ hiếm** — nhóm 1 đã xuống 0,35, nhóm 7 vẫn ở 3,3, gần mức ban đầu 4,84.
Adam: mọi nhóm đều dưới 0,15, và nhóm hiếm còn thấp hơn nhóm phổ biến. Sign descent đảo ngược hẳn thứ tự: nhóm hiếm nhất
đã ở 0,009, trong khi nhóm phổ biến nhất còn ở 0,88, cao hơn cả gradient descent. Nó không nhanh hơn ở mọi nơi; nó chỉ
không bỏ rơi các lớp hiếm.

Vì sao? Gradient của mất mát theo các trọng số của lớp $k$ tỉ lệ với tần suất của lớp đó, nên lớp hiếm nhận gradient
nhỏ. Gradient descent đi một bước tỉ lệ với gradient, và $\eta$ của nó bị giới hạn bởi lớp phổ biến nhất (Bài 05: một
bước lớn hơn $2/\lambda$ theo hướng có độ cong $\lambda$ lớn nhất sẽ nổ). Adam và sign descent chia mỗi gradient cho
độ lớn của chính nó, nên mỗi lớp đi những bước cỡ như nhau, bất kể nó hiếm hay phổ biến. Hình dưới là 3 seed đầy đủ,
500 bước.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3), sharey=True)
steps_logged = TAIL["check"]
cmap = plt.get_cmap("viridis")
for ax, o in zip(axes, ("gd", "sign", "adam")):
    curves = np.array(TAIL["heavy"]["runs"][o])  # (seed, check, group)
    med = np.median(curves, 0)
    for k in range(M_GROUPS):
        ax.plot(steps_logged, np.maximum(med[:, k], 1e-4), color=cmap(k / (M_GROUPS - 1)),
                ls="-" if k % 2 == 0 else "--", lw=1.6, label=f"nhóm {k + 1}")  # fmt: skip
    ax.set(xscale="log", yscale="log", title=f"{names[o]} (η = {num(best[o], 2)})", xlabel="bước")
    ax.set_ylim(1e-4, 10)
axes[0].set_ylabel("mất mát của nhóm (trung vị 3 seed)")
axes[2].legend(fontsize=7, frameon=False, ncol=2, loc="lower left")
vn_axes(fig)
plt.tight_layout()
plt.show()

Mỗi khung là một bộ tối ưu, trục ngang là số bước, trục dọc là mất mát của từng nhóm (cả hai trục thang log). Màu đi
từ tím (nhóm 1, phổ biến nhất) đến vàng (nhóm 7, hiếm nhất), nét liền và nét đứt xen kẽ để phân biệt các nhóm kề
nhau. Với gradient descent, các đường xếp tầng: nhóm càng hiếm càng ở cao. Với sign descent và Adam, thứ tự đảo ngược:
nhóm hiếm xuống trước và xuống thấp nhất.

**Đối chứng**: cùng bài toán nhưng 127 lớp **đều nhau** (7 mẫu mỗi lớp). Nếu khoảng cách đến từ đuôi dài, nó phải
thu hẹp khi bỏ đuôi đi:

In [ ]:
for dist, label in (("heavy", "tần suất lệch"), ("uniform", "tần suất đều")):
    finals = {o: [run[-1] for run in TAIL[dist]["runs"][o]] for o in ("gd", "sign", "adam")}
    tot = {o: [float(np.mean(f)) for f in fs] for o, fs in finals.items()}
    print(f"{label:13}: " + " · ".join(f"{names[o]} η {num(TAIL[dist]['best'][o], 2)}: mất mát {num(min(tot[o]))}–"
                                         f"{num(max(tot[o]))}" for o in tot))  # fmt: skip

Khi các lớp đều nhau, gradient descent chịu được $\eta$ lớn gấp 10 lần (1 thay vì 0,1), và mất mát cuối của nó giảm từ
khoảng 0,97 xuống 0,028. Adam vẫn thấp hơn, nhưng khoảng cách tuyệt đối gần như biến mất. Cái làm gradient descent chậm
không phải là bài toán khó, mà là **một lớp phổ biến buộc nó đi bước nhỏ, trong khi các lớp hiếm cần bước lớn**.

**Kết luận: khớp.** Trên đúng bài toán của bài báo, gradient descent tụt hậu đúng ở các lớp hiếm, còn Adam và sign
descent thì không; đối chứng tần suất đều thu hẹp hẳn khoảng cách. Giới hạn: đây là một mô hình tuyến tính trên dữ liệu ngẫu
nhiên; bài báo dùng thêm một transformer đóng băng trên văn bản thật, mà notebook này không chạy lại.

## 4 · Rìa ổn định: độ cong tự dừng ở $2/\eta$

**Nhận định** (Cohen và cộng sự, 2021): Bài 05 nói gradient descent với bước $\eta$ chỉ ổn định khi **độ sắc**
(sharpness) $S$, trị riêng lớn nhất của ma trận Hessian của hàm mất mát, nhỏ hơn $2/\eta$. Với mạng nơ-ron huấn luyện
toàn lô, người ta thấy $S$ ban đầu **tăng dần** ("progressive sharpening") cho tới đúng $2/\eta$, rồi **dừng lơ lửng ở
đó** — trong khi mất mát vẫn giảm, dù không còn giảm đều mỗi bước. Mạng tự đưa mình về rìa của vùng ổn định. Cohen và
cộng sự (2024) sau đó đề xuất "central flows", một phương trình dự đoán quỹ đạo trung bình trong chế độ này; notebook
chỉ kiểm hiện tượng, không kiểm lý thuyết đó.

Thí nghiệm: MLP 784 → 128 → 128 → 10 với tanh, mất mát bình phương $\tfrac12\lVert f(x) - y\rVert^2$ (như thí nghiệm chính
của Cohen; với cross-entropy, độ sắc giảm dần về cuối), 1.000 ảnh MNIST, gradient descent toàn lô. $S$ đo bằng **lặp
lũy thừa** trên tích Hessian–vectơ: không cần dựng Hessian (hơn 100 nghìn × 100 nghìn phần tử), chỉ cần hỏi "Hessian
nhân với vectơ $v$ bằng bao nhiêu", việc mà PyTorch làm bằng hai lượt lan truyền ngược.

In [ ]:
def mnist_first(n=1000):
    with gzip.open(DATA / "train-images-idx3-ubyte.gz") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)[:n]
    with gzip.open(DATA / "train-labels-idx1-ubyte.gz") as f:
        lab = np.frombuffer(f.read(), np.uint8, offset=8)[:n]
    return torch.from_numpy(x.astype(np.float32) / 255), nn.functional.one_hot(
        torch.from_numpy(lab.astype(np.int64)), 10
    ).float()


Xm, Ym = mnist_first()


def mlp_tanh(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(784, 128), nn.Tanh(), nn.Linear(128, 128), nn.Tanh(), nn.Linear(128, 10))


def mse(m):
    return ((m(Xm) - Ym) ** 2).sum(1).mean() / 2


def sharpness(m, v=None, iters=20):
    """Trị riêng lớn nhất của Hessian bằng lặp lũy thừa: v ← Hv / ‖Hv‖, lặp lại. Trả về (S, v)."""
    ps = list(m.parameters())
    g = torch.autograd.grad(mse(m), ps, create_graph=True)
    v = [torch.randn_like(p) for p in ps] if v is None else v
    for _ in range(iters):
        norm = torch.sqrt(sum((a * a).sum() for a in v))
        v = [a / norm for a in v]
        hv = torch.autograd.grad(g, ps, v, retain_graph=True)
        lam = sum((a * b).sum() for a, b in zip(hv, v)).item()
        v = [h.detach() for h in hv]
    return lam, v


def gd_run(eta, steps, seed=0, every=25):
    """Gradient descent toàn lô; S mỗi `every` bước (khởi động từ vectơ lần trước), mất mát mỗi bước."""
    m = mlp_tanh(seed)
    opt = torch.optim.SGD(m.parameters(), lr=eta)
    loss_curve, sharp, v = [], [], None
    for s in range(steps + 1):
        if s % every == 0:
            lam, v = sharpness(m, v)
            sharp.append(lam)
        opt.zero_grad()
        loss = mse(m)
        loss_curve.append(loss.item())
        if s < steps:
            loss.backward()
            opt.step()
    return loss_curve, sharp


n_params = sum(p.numel() for p in mlp_tanh(0).parameters())
print(f"{n_params:,} tham số".replace(",", "."))

### 🤔 Dự đoán trước
Với $\eta = 0{,}1$, giới hạn ổn định cổ điển là $2/\eta = 20$. Lúc khởi tạo $S \approx 9{,}5$. Sau vài trăm bước, $S$
sẽ (a) vẫn quanh 9,5, (b) vượt hẳn 20 và mạng nổ, hay (c) dừng lại quanh 20?

In [ ]:
loss_live, sharp_live = gd_run(0.1, 400)
print("bước   S      mất mát")
for s in (0, 100, 200, 300, 400):
    print(f"{s:4}  {vn(sharp_live[s // 25], 1):>5}  {vn(loss_live[s], 4)}")
stored = EOS["runs"]["0.1"][0]
assert abs(sharp_live[300 // 25] - stored["sharp"][300 // 25]) < 1e-2 * stored["sharp"][300 // 25], (
    "khác với kết quả đã lưu"
)
assert abs(loss_live[400] - stored["loss"][400 // EOS["lossEvery"]]) < 1e-3, "khác với kết quả đã lưu"
ups = sum(b > a for a, b in pairwise(loss_live))
print(f"số bước mà mất mát tăng lên: {ups}/400")

$S$ tăng từ 9,5 lên quá 20 một chút rồi dao động quanh đó: đúng (c). Và kể từ đó mất mát không còn giảm đều; nó tăng
ở một số bước, dù xu hướng chung vẫn đi xuống. Kết quả đầy đủ: 4 tốc độ học × 3 seed × 3.000 bước.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
steps_s = np.arange(0, EOS["steps"] + 1, EOS["every"])
steps_l = np.arange(0, EOS["steps"] + 1, EOS["lossEvery"])
colors = {"0.02": GREY, "0.05": GREEN, "0.1": BLUE, "0.2": ORANGE}
styles = {"0.02": ":", "0.05": "-.", "0.1": "-", "0.2": "--"}
for e, runs in EOS["runs"].items():
    eta = float(e)
    S = np.median([r_["sharp"] for r_ in runs], 0)
    L = np.median([r_["loss"] for r_ in runs], 0)
    axes[0].plot(steps_s, S, color=colors[e], ls=styles[e], lw=1.8, label=f"η = {vn(eta, 2)}")
    axes[0].axhline(2 / eta, color=colors[e], lw=0.8, alpha=0.6)
    axes[1].plot(steps_l, L, color=colors[e], ls=styles[e], lw=1.4, label=f"η = {vn(eta, 2)}")
axes[0].set(xlabel="bước", ylabel="độ sắc S (trung vị 3 seed)", ylim=(0, 110), title="S và các vạch 2/η")
axes[1].set(xlabel="bước", ylabel="mất mát (trung vị 3 seed)", yscale="log", title="Mất mát")
axes[0].legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.tight_layout()
plt.show()

In [ ]:
for e, runs in EOS["runs"].items():
    eta = float(e)
    end = [r_["sharp"][-1] for r_ in runs]
    late = [float(np.mean(r_["sharp"][len(r_["sharp"]) // 2 :])) for r_ in runs]
    final = [r_["loss"][-1] for r_ in runs]
    ups = [sum(b > a for a, b in pairwise(r_["loss"])) for r_ in runs]
    print(f"η = {vn(eta, 2):>4}: 2/η = {vn(2 / eta, 0):>3} · S trung bình nửa sau {vn(min(late), 1)}–{vn(max(late), 1)}, "
          f"ở bước cuối {vn(min(end), 1)}–{vn(max(end), 1)} · "
          f"mất mát cuối {vn(min(final), 3)}–{vn(max(final), 3)} · điểm tăng (mỗi 5 bước) {min(ups)}–{max(ups)}/600")  # fmt: skip

Bên trái, mỗi đường là $S$ của một tốc độ học (màu và kiểu nét khác nhau), mỗi vạch ngang mảnh cùng màu là $2/\eta$
của nó. Với $\eta = 0{,}05$, 0,1 và 0,2, $S$ leo lên rồi nằm sát vạch của mình (trung bình nửa sau 40,3–40,7, 20,7–20,9
và 10,5–10,7, so với 40, 20 và 10). Với $\eta = 0{,}02$, vạch ở 100 và $S$ vẫn đang tăng khi hết 3.000 bước, còn xa
vạch: mạng vẫn ở chế độ cổ điển, và mất mát giảm đều ở mọi điểm đo. Bên phải: tốc độ học càng lớn thì mất mát cuối càng **thấp** — chạy ở rìa ổn định
không phải là hỏng, đó là chế độ làm việc bình thường của gradient descent với $\eta$ lớn.

**Kết luận: khớp.** Độ sắc tự tăng tới $2/\eta$ rồi dừng ở đó, ở cả ba tốc độ học đủ lớn và cả 3 seed. Quy tắc
"chọn $\eta < 2/S$" của Bài 05 vẫn đúng cho từng bước, nhưng trong mạng nơ-ron $S$ không cố định: nó thay đổi theo
chính $\eta$ mà ta chọn.

## 5 · Grokking và sụp đổ softmax

**Nhận định.** Trên các phép toán modulo, một mạng có thể đạt 100% trên tập huấn luyện từ rất sớm mà tập kiểm tra
vẫn ở mức đoán mò, rồi rất lâu sau mới đột ngột tổng quát hóa: **grokking** (Power và cộng sự, 2022). Thường nó chỉ
xảy ra khi có weight decay. Prieto và cộng sự (2025) giải thích vì sao không có weight decay thì grokking hay không
đến:
1. Khi mọi mẫu huấn luyện đã đúng, cách rẻ nhất để giảm cross-entropy là **phóng to các logit** theo hướng chúng đang
   chỉ: bước cập nhật gần như song song với chính các trọng số ("giảm mất mát ngây thơ").
2. Logit lớn tới mức softmax dạng float32 làm tròn xác suất của lớp đúng thành **đúng 1**: mất mát và gradient của mẫu
   đó bằng đúng 0, và việc học dừng lại. Họ gọi đó là **sụp đổ softmax**.

Hai cách chữa của họ: **StableMax**, một softmax mà $s(x)$ tăng tuyến tính thay vì theo hàm mũ, và **⊥Grad**, bỏ phần
gradient song song với trọng số trước mỗi bước. Thí nghiệm dùng đúng thiết lập trong mã nguồn của họ: $(a + b) \bmod
113$, 40% các cặp để huấn luyện, MLP 226 → 200 → 200 → 113 không có bias, AdamW toàn lô với $\varepsilon = 10^{-25}$.

Mỗi thí nghiệm dưới đây chạy 3 seed; seed quyết định cách chia 40/60, trọng số khởi tạo và thứ tự xáo trộn mỗi epoch.

In [ ]:
P_MOD = 113


def mod_data(seed):
    """(a + b) mod 113, mọi cặp, 40% để huấn luyện; giống hệt reel/research/grok.py."""
    g = torch.Generator().manual_seed(seed)
    a, b = torch.meshgrid(torch.arange(P_MOD), torch.arange(P_MOD), indexing="ij")
    a, b = a.flatten(), b.flatten()
    y = (a + b) % P_MOD
    X = torch.cat([nn.functional.one_hot(a, P_MOD), nn.functional.one_hot(b, P_MOD)], 1).float()
    perm = torch.randperm(P_MOD * P_MOD, generator=g)
    n = int(0.4 * P_MOD * P_MOD)
    return X[perm[:n]], y[perm[:n]], X[perm[n:]], y[perm[n:]]


def mod_mlp(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2 * P_MOD, 200, bias=False), nn.ReLU(), nn.Linear(200, 200, bias=False), nn.ReLU(),
                         nn.Linear(200, P_MOD, bias=False))  # fmt: skip


def orthogonalise(params):
    """⊥Grad (mã của tác giả): bỏ phần gradient song song với trọng số, rồi giữ nguyên độ dài gradient."""
    with torch.no_grad():
        for q in params:
            w, g = q.view(-1), q.grad.view(-1)
            go = g - (w @ g) / (w @ w + 1e-30) * w
            q.grad.copy_((go * (g.norm() / (go.norm() + 1e-30))).view_as(q.grad))


def grok_run(ortho, epochs=200, seed=0, lr=0.01):
    Xtr, ytr, Xte, yte = mod_data(seed)
    m = mod_mlp(seed)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, betas=(0.9, 0.99), eps=1e-25, weight_decay=0.0)
    torch.manual_seed(10_000 + seed)  # thứ tự xáo trộn mỗi epoch
    log = []
    for e in range(epochs + 1):
        if e % 50 == 0:
            with torch.no_grad():
                log.append((e, (m(Xtr).argmax(1) == ytr).float().mean().item(),
                            (m(Xte).argmax(1) == yte).float().mean().item()))  # fmt: skip
        if e == epochs:
            break
        perm = torch.randperm(len(Xtr))
        loss = nn.functional.cross_entropy(m(Xtr[perm]), ytr[perm])
        opt.zero_grad()
        loss.backward()
        if ortho:
            orthogonalise(m.parameters())
        opt.step()
    return log


Xtr, ytr, Xte, yte = mod_data(0)
print(f"{vn(len(ytr), 0)} cặp để huấn luyện, {vn(len(yte), 0)} cặp để kiểm tra")

### 🤔 Dự đoán trước
Huấn luyện 200 epoch, không weight decay, bằng AdamW thường và bằng AdamW với ⊥Grad. Sau 200 epoch, mạng nào
đúng trên tập kiểm tra: không mạng nào, chỉ ⊥Grad, hay cả hai?

In [ ]:
live = {name: grok_run(ortho) for name, ortho in (("AdamW thường", False), ("AdamW + ⊥Grad", True))}
for name, log in live.items():
    print(
        f"{name:14}: " + " · ".join(f"epoch {e}: huấn luyện {pct(a, 0)}, kiểm tra {pct(b, 1)}" for e, a, b in log[1::2])
    )
for (name, log), key in zip(live.items(), ("softmax-lr0.01", "ortho-lr0.01")):
    stored = GROK["live"]["runs"][key]
    assert abs(log[-1][2] - stored["testAcc"][-1]) < 0.05, f"{name}: khác với kết quả đã lưu"

AdamW thường thuộc lòng tập huấn luyện ngay từ epoch 50 nhưng đoán mò trên tập kiểm tra. Với ⊥Grad, mạng học
chậm hơn trên tập huấn luyện, rồi ở epoch 150 đúng gần như mọi cặp **chưa từng thấy**. Kết quả đầy đủ, 3 seed:

In [ ]:
LABEL = {
    "softmax-lr0.005": "softmax, float32, η 0,005", "softmax-lr0.01": "softmax, float32, η 0,01",
    "softmax-f64": "softmax, float64", "stablemax": "StableMax (40.000 epoch)",
    "stablemax-80k": "StableMax (80.000 epoch)", "ortho-lr0.005": "⊥Grad, η 0,005", "ortho-lr0.01": "⊥Grad, η 0,01",
    "wd1": "weight decay 1",
}  # fmt: skip


def epochs_text(es):
    """[150, 150, None] → "150, 150, không" (không = không bao giờ trong thời gian chạy)."""
    return ", ".join("không" if e is None else vn(e, 0) for e in es)


for a, runs in GROK["runs"].items():
    grok_at = [r_["grokEpoch"] for r_ in runs]
    first_zero = [next((e for e, z in zip(r_["epoch"], r_["zero"]) if z > 0.01), None) for r_ in runs]
    print(f"{LABEL[a]:26} kiểm tra cuối {pct(min(r_['testAcc'][-1] for r_ in runs), 1)}–"
          f"{pct(max(r_['testAcc'][-1] for r_ in runs), 1)} · đạt 95% ở epoch {epochs_text(grok_at)} · "
          f"mất mát đúng bằng 0 ở tối đa {pct(max(max(r_['zero']) for r_ in runs), 0)} mẫu "
          f"(từ epoch {epochs_text(first_zero)}) · cos tối đa "
          f"{vn(max(max(c for c in r_['cos'] if c is not None) for r_ in runs), 2)}")  # fmt: skip

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
show = [("softmax-lr0.01", GREY, ":"), ("softmax-f64", ORANGE, "--"), ("wd1", GREEN, "-."), ("ortho-lr0.01", BLUE, "-")]
show.append(("stablemax-80k" if "stablemax-80k" in GROK["runs"] else "stablemax", "#7a4fb5", (0, (5, 1, 1, 1))))
for a, col, ls in show:
    runs = GROK["runs"][a]
    ep = np.array(runs[0]["epoch"], float)
    ep[0] = 1  # trục log: epoch 0 vẽ ở 1
    axes[0].plot(ep, np.median([r_["testAcc"] for r_ in runs], 0), color=col, ls=ls, lw=1.8, label=LABEL[a])
    if a.startswith("softmax") or a == "wd1":
        axes[1].plot(ep, np.median([r_["zero"] for r_ in runs], 0), color=col, ls=ls, lw=1.8, label=LABEL[a])
axes[0].set(xscale="log", xlabel="epoch", ylabel="độ chính xác kiểm tra (trung vị 3 seed)", title="Tổng quát hóa")
axes[1].set(xscale="log", xlabel="epoch", ylabel="tỉ lệ mẫu huấn luyện có mất mát đúng bằng 0",
            title="Sụp đổ softmax")  # fmt: skip
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
axes[0].legend(frameon=False, fontsize=8)
axes[1].legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.tight_layout()
plt.show()

Bên trái là độ chính xác trên tập kiểm tra theo epoch (trục ngang thang log), mỗi cách huấn luyện một màu và một
kiểu nét. Bên phải là tỉ lệ mẫu huấn luyện có mất mát **đúng bằng 0** trong số thực: dấu hiệu của sụp đổ softmax.

- **Softmax thường, float32:** thuộc lòng tập huấn luyện trong 50–100 epoch, rồi bước cập nhật gần như song song với
  chính trọng số ("cos tối đa" ở ô trên: cosin giữa −gradient thô và trọng số; gần 1 nghĩa là bước cập nhật chủ yếu
  phóng to logit). Từ khoảng epoch 1.600, các mẫu bắt đầu có mất mát đúng 0; tỉ lệ đó dao động mạnh, có lúc lên tới
  89–97% tập huấn luyện. Tập kiểm tra không bao giờ rời mức đoán mò.
- **Float64:** sụp đổ đến muộn hơn nhiều (từ khoảng epoch 5.500) và ít hơn, nhưng vẫn không grokking trong 6.000
  epoch: với 40% dữ liệu, bài báo cũng thấy sụp đổ đến trước khi tập kiểm tra kịp nhích lên, kể cả với float64.
- **⊥Grad:** gradient thô của nó cũng gần song song với trọng số (cos tới 0,84–0,94, đo trước khi chiếu), nhưng phần
  đó bị bỏ đi trước mỗi bước. Không bao giờ có mẫu nào mất mát bằng 0, và grokking ở epoch 150–200 trên cả 3 seed, mà
  không cần weight decay.
- **Weight decay:** grokking ở epoch 1.000. Cũng có tới khoảng 30% mẫu mất mát bằng 0 ở giữa chừng, nhưng weight decay
  kéo các logit về, nên phần còn lại của tập huấn luyện vẫn cho gradient.
- **StableMax:** cũng grokking, nhưng chậm hơn ⊥Grad rất nhiều. Sau 40.000 epoch, tập kiểm tra mới ở 12–59%; sau
  80.000 epoch (độ dài trong mã của tác giả), 2 trong 3 seed vượt 95% (ở epoch 60.000 và 66.400), và cả ba kết thúc ở
  90,6–98,8%. Không mẫu nào có mất mát bằng 0: StableMax tăng tuyến tính chứ không theo hàm mũ, nên không bị làm tròn.

**Kết luận: khớp, với một điều kiện.** Không có weight decay, softmax float32 sụp đổ và không grokking; cả hai cách
chữa của bài báo đều grokking không cần weight decay: ⊥Grad rất nhanh, StableMax rất chậm. Nhưng các kết quả này nhạy
với siêu tham số: lần thử đầu khi chuẩn bị notebook này (modulo 59, có bias, $\varepsilon = 10^{-8}$ mặc định của Adam)
không thấy ⊥Grad hay StableMax giúp gì. Chỉ khi dùng đúng thiết lập trong mã nguồn của tác giả thì hiện tượng mới hiện
ra rõ.

## Tóm tắt
- **Adam và lớp hiếm — khớp.** Trên bài toán của Kunstner và cộng sự, gradient descent để lại các lớp hiếm ở mất mát
  cao (nhóm hiếm nhất 3,3 sau 300 bước), còn Adam đưa mọi nhóm xuống dưới 0,15; khi các lớp đều nhau, khoảng cách thu
  hẹp hẳn.
- **Rìa ổn định — khớp.** Độ sắc tự tăng tới $2/\eta$ rồi dừng ở đó với $\eta$ 0,05, 0,1 và 0,2, ở cả 3 seed; mất mát vẫn
  giảm, dù không đều.
- **Grokking và sụp đổ softmax — khớp, nhưng nhạy với thiết lập.** Softmax float32 không weight decay sụp đổ và không
  bao giờ tổng quát hóa; ⊥Grad grokking sau 150–200 epoch, weight decay sau 1.000, StableMax sau 60.000–66.400 (2 trong
  3 seed, trong 80.000 epoch).
- Cả ba kết quả đều xuất hiện ở quy mô nhỏ đến mức chạy được trên CPU. Điều đó không chứng minh chúng đúng ở mô hình
  lớn, nhưng cho thấy cơ chế đứng sau chúng không cần mô hình lớn mới có.

## Câu hỏi ôn tập
1. Vì sao gradient của các trọng số của một lớp tỉ lệ với tần suất của lớp đó, và điều đó làm gradient descent chậm
   ở đâu?
2. Sign descent không dùng độ lớn của gradient. Vì sao nó lại thắng gradient descent trên bài toán này?
3. Ở rìa ổn định, mất mát không giảm ở mọi bước. Vì sao điều đó không mâu thuẫn với việc nó vẫn giảm về lâu dài?
4. Vì sao với $\eta = 0{,}02$ ta không thấy rìa ổn định trong 3.000 bước?
5. "Sụp đổ softmax" là gì, và vì sao float64 làm nó đến muộn hơn?
6. ⊥Grad bỏ phần nào của gradient, và vì sao bỏ phần đó lại giúp mạng tổng quát hóa?

<details><summary>Gợi ý đáp án</summary>

1. Mất mát là trung bình trên các mẫu, và chỉ các mẫu của lớp $k$ đẩy mạnh logit của lớp $k$ lên; lớp có ít mẫu thì
   tổng đóng góp nhỏ. Gradient descent đi bước tỉ lệ với gradient, mà $\eta$ đã bị lớp phổ biến nhất giới hạn, nên lớp
   hiếm đi rất chậm.
2. Nó đi một bước bằng nhau cho mọi tham số, nên các trọng số của lớp hiếm tiến nhanh như của lớp phổ biến.
3. Mạng dao động qua lại theo hướng sắc nhất, nhưng vẫn tiến dần theo các hướng khác; Cohen và cộng sự (2024) mô tả
   quỹ đạo trung bình đó bằng "central flows".
4. $2/\eta = 100$, còn độ sắc vẫn đang tăng dần và còn xa 100 khi hết 3.000 bước: mạng chưa chạm tới rìa.
5. Logit lớn tới mức số thực float32 làm tròn xác suất của lớp đúng thành đúng 1, nên mất mát và gradient của mẫu đó
   bằng đúng 0. Float64 có nhiều chữ số hơn, nên phải đợi logit lớn hơn nữa.
6. Phần song song với trọng số, tức phần chỉ phóng to các logit mà không đổi dự đoán. Bỏ nó đi, mỗi bước buộc phải đổi
   hướng của trọng số, tức đổi cách mạng tính, và mất mát không thể giảm chỉ bằng cách phóng to.
</details>

## Tìm hiểu thêm
- F. Kunstner và cộng sự (2024), [Heavy-Tailed Class Imbalance and Why Adam Outperforms Gradient Descent on Language
  Models](https://arxiv.org/abs/2402.19449); F. Kunstner & F. Bach (2025), [Scaling Laws for Gradient Descent and Sign
  Descent for Linear Bigram Models under Zipf's Law](https://arxiv.org/abs/2505.19227); R. Yadav và cộng sự (2025),
  [Provable Benefit of SignGD](https://arxiv.org/abs/2512.00763).
- J. Cohen và cộng sự (2021), [Gradient Descent on Neural Networks Typically Occurs at the Edge of
  Stability](https://arxiv.org/abs/2103.00065); J. Cohen và cộng sự (2024), [Understanding Optimization in Deep
  Learning with Central Flows](https://arxiv.org/abs/2410.24206).
- L. Prieto và cộng sự (2025), [Grokking at the Edge of Numerical Stability](https://arxiv.org/abs/2501.04697) và mã
  nguồn của họ (thiết lập mà mục 5 dùng); A. Power và cộng sự (2022),
  [Grokking: Generalization Beyond Overfitting on Small Algorithmic Datasets](https://arxiv.org/abs/2201.02177).

**Quay lại:** [Bài 07 · Momentum, Adam & AdamW](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/).

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · torch {torch.__version__} · numpy {np.__version__} · "
      f"matplotlib {matplotlib.__version__}")  # fmt: skip